# Task II — Classical Graph Neural Networks for Quark/Gluon Jet Classification

**ML4SCI GSoC 2026 Evaluation Test**

Two graph-based architectures for classifying jets as quark- or gluon-initiated,
using the ParticleNet dataset (Komiske, Metodiev & Thaler, 2019 — Zenodo 3164691).

### Architectures
1. **Graph Convolutional Network (GCN)** — particles as nodes, kNN edges in (Δη, Δφ) space.
2. **TreeNiN-style RvNN** (Louppe, Cho, Cranmer et al. JHEP 2019) — kt jet clustering tree
   processed bottom-up with level-batching for GPU acceleration. GRU cell at internal nodes
   with IRC-safe splitting features (log kt distance, momentum fraction z).

### Graph Construction Considerations
- **Node features**: log(pT), log(pT/pT_jet), Δη, Δφ, pdgid — IRC-safe relative coordinates.
- **GCN edges**: ΔR kNN (k=7). Data-driven geometric connectivity without assuming shower history.
- **RvNN tree**: kt algorithm (p=−1) in pure numpy. IRC-safe merging history mirrors QCD shower.
  Level-batching processes all nodes at the same depth simultaneously — real GPU parallelism.

### Dependencies
Pure PyTorch only — no torch-geometric, no pyjet. Compatible with Colab numpy 2.x.

## 0. Installation

In [ ]:
!pip install -q energyflow

## 1. Imports & Seeds

In [ ]:
import numpy as np
import energyflow as ef
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from collections import deque
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings, random, os, pickle
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | Torch: {torch.__version__} | Numpy: {np.__version__}')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ml4sci_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive ready: {DRIVE_DIR}')
for f in os.listdir(DRIVE_DIR):
    print(f'  {f}: {os.path.getsize(f"{DRIVE_DIR}/{f}")/1e6:.1f} MB')

## 3. Data Loading & EDA

In [ ]:
X, y = ef.qg_jets.load(num_data=20000, pad=True, cache_dir=DRIVE_DIR)
print(f'X: {X.shape}  y: {y.shape}  quarks: {y.mean():.3f}')

In [ ]:
mult  = (X[:,:,0]>0).sum(1)
pt_fr = X[:,0,0]/(X[:,:,0].sum(1)+1e-8)
eta   = X[:,:,1]; phi=X[:,:,2]; mask=X[:,:,0]>0
em    = (eta*mask).sum(1)/(mask.sum(1)+1e-8)
pm    = (phi*mask).sum(1)/(mask.sum(1)+1e-8)
dR    = np.sqrt(((eta-em[:,None])**2+(phi-pm[:,None])**2))
mdr   = (dR*mask).sum(1)/(mask.sum(1)+1e-8)

fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,d,xl,tl in zip(axes,[mult,pt_fr,mdr],
    ['Multiplicity','Leading pT fraction','Mean Delta-R'],
    ['Constituent multiplicity','pT fraction','Angular spread']):
    ax.hist(d[y==1],bins=40,alpha=0.6,label='Quark',color='steelblue',density=True)
    ax.hist(d[y==0],bins=40,alpha=0.6,label='Gluon',color='tomato',density=True)
    ax.set_xlabel(xl); ax.set_title(tl); ax.legend()
plt.suptitle('Quark vs Gluon jet properties',y=1.01)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/eda.png',dpi=150,bbox_inches='tight')
plt.show()
print('Gluon: higher multiplicity + larger spread. Consistent with QCD Ca>Cf.')

## 4. Shared Preprocessing
IRC-safe features per particle: log(pT), log(pT/pT_jet), Δη, Δφ, pdgid.
Capped at N_MAX=50 constituents.

In [ ]:
N_MAX    = 50
N_EPOCHS = 30
PDGID_MAP= {0:0,22:1,211:2,-211:2,321:3,-321:3,130:4,
             2112:5,-2112:5,2212:6,-2212:6,11:7,-11:7,13:8,-13:8}

def preprocess_jets(X, n_max=N_MAX):
    jets=[]
    for jet in X:
        p=jet[jet[:,0]>0][:n_max]
        if len(p)<3: jets.append(None); continue
        pt,eta,phi,pid=p[:,0],p[:,1],p[:,2],p[:,3]
        ps=(pt.sum()); deta=eta-(pt*eta).sum()/ps
        dphi=(phi-(pt*phi).sum()/ps+np.pi)%(2*np.pi)-np.pi
        jets.append(np.stack([
            np.log(pt+1e-8),np.log(pt/(ps+1e-8)+1e-8),
            deta,dphi,
            np.array([PDGID_MAP.get(int(q),0) for q in pid],dtype=np.float32)
        ],axis=1).astype(np.float32))
    return jets

jet_features=preprocess_jets(X)
valid=[(f,int(y[i])) for i,f in enumerate(jet_features) if f is not None]
feats_list=[v[0] for v in valid]
labels_list=[v[1] for v in valid]
print(f'Valid: {len(valid)}/{len(X)}')

idx=list(range(len(valid)))
idx_train,idx_tmp=train_test_split(idx,test_size=0.30,random_state=SEED,stratify=labels_list)
lt=[labels_list[i] for i in idx_tmp]
idx_val,idx_test=train_test_split(idx_tmp,test_size=0.50,random_state=SEED,stratify=lt)
print(f'Train:{len(idx_train)} Val:{len(idx_val)} Test:{len(idx_test)}')

---
## 5. Architecture 1 — Graph Convolutional Network (GCN)

**Graph construction**: kNN (k=7) in (Δη, Δφ) space using ΔR metric.
Pure PyTorch — no torch-geometric required.

**Model**: 3-layer GCN with residual connections.
Global readout: concat(mean_pool, max_pool) → MLP classifier.
Permutation-invariant via global pooling.

In [ ]:
def build_knn_adj(pos,k=7):
    N=pos.shape[0]
    d=(pos.unsqueeze(0)-pos.unsqueeze(1)).pow(2).sum(-1)
    d.fill_diagonal_(float('inf'))
    _,idx=d.topk(min(k,N-1),dim=1,largest=False)
    src=torch.arange(N).unsqueeze(1).expand_as(idx).reshape(-1)
    dst=idx.reshape(-1)
    ei=torch.stack([torch.cat([src,dst]),torch.cat([dst,src])],0)
    return torch.unique(ei,dim=1)

class GCNLayer(nn.Module):
    def __init__(self,i,o): super().__init__(); self.lin=nn.Linear(i,o)
    def forward(self,x,ei,N):
        sl=torch.arange(N,device=x.device).unsqueeze(0).expand(2,-1)
        ei=torch.cat([ei,sl],1)
        deg=torch.zeros(N,device=x.device)
        deg.scatter_add_(0,ei[0],torch.ones(ei.shape[1],device=x.device))
        agg=torch.zeros_like(x)
        agg.scatter_add_(0,ei[1].unsqueeze(1).expand(-1,x.shape[1]),x[ei[0]])
        return self.lin(agg/(deg.clamp(min=1).unsqueeze(1)))

class GCNJetClassifier(nn.Module):
    def __init__(self,in_d=5,h=128,out_d=2,drop=0.3):
        super().__init__()
        self.proj=nn.Linear(in_d,h)
        self.c1,self.c2,self.c3=GCNLayer(h,h),GCNLayer(h,h),GCNLayer(h,h)
        self.b1,self.b2,self.b3=[nn.BatchNorm1d(h) for _ in range(3)]
        self.drop=nn.Dropout(drop)
        self.mlp=nn.Sequential(
            nn.Linear(2*h,128),nn.ReLU(),nn.Dropout(drop),
            nn.Linear(128,64),nn.ReLU(),nn.Linear(64,out_d))
    def _enc(self,x,ei):
        N=x.shape[0]; x=F.relu(self.proj(x))
        r=x; x=F.relu(self.b1(self.c1(x,ei,N))+r); x=self.drop(x)
        r=x; x=F.relu(self.b2(self.c2(x,ei,N))+r); x=self.drop(x)
        r=x; x=F.relu(self.b3(self.c3(x,ei,N))+r)
        return x
    def forward(self,bf,be):
        out=[torch.cat([self._enc(x,ei).mean(0),self._enc(x,ei).max(0).values])
             for x,ei in zip(bf,be)]
        return self.mlp(torch.stack(out))

class GCNDataset(Dataset):
    def __init__(self,idx): self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=self.idx[i]; f=torch.tensor(feats_list[j],dtype=torch.float)
        return f,build_knn_adj(f[:,2:4]),labels_list[j]

def gcn_col(b): return [x[0] for x in b],[x[1] for x in b],torch.tensor([x[2] for x in b],dtype=torch.long)

gcn_tr=DataLoader(GCNDataset(idx_train),128,shuffle=True, collate_fn=gcn_col,num_workers=0)
gcn_vl=DataLoader(GCNDataset(idx_val),  128,shuffle=False,collate_fn=gcn_col,num_workers=0)
gcn_te=DataLoader(GCNDataset(idx_test), 128,shuffle=False,collate_fn=gcn_col,num_workers=0)
gcn_model=GCNJetClassifier().to(DEVICE)
print(f'GCN params: {sum(p.numel() for p in gcn_model.parameters()):,}')

In [ ]:
def train_gcn(model,loader,opt,crit):
    model.train(); tl=tc=tt=0
    for f,e,l in loader:
        f=[x.to(DEVICE) for x in f]; e=[x.to(DEVICE) for x in e]; l=l.to(DEVICE)
        opt.zero_grad(); lg=model(f,e); loss=crit(lg,l); loss.backward(); opt.step()
        tl+=loss.item()*len(l); tc+=(lg.argmax(1)==l).sum().item(); tt+=len(l)
    return tl/tt,tc/tt

@torch.no_grad()
def eval_gcn(model,loader,crit):
    model.eval(); tl=tc=tt=0; P,L=[],[]
    for f,e,l in loader:
        f=[x.to(DEVICE) for x in f]; e=[x.to(DEVICE) for x in e]; l=l.to(DEVICE)
        lg=model(f,e); loss=crit(lg,l)
        tl+=loss.item()*len(l); tc+=(lg.argmax(1)==l).sum().item(); tt+=len(l)
        P.append(F.softmax(lg,1)[:,1].cpu()); L.append(l.cpu())
    P=torch.cat(P).numpy(); L=torch.cat(L).numpy()
    return tl/tt,tc/tt,roc_auc_score(L,P),P,L

gcn_opt=torch.optim.AdamW(gcn_model.parameters(),lr=3e-4,weight_decay=1e-4)
gcn_sch=torch.optim.lr_scheduler.CosineAnnealingLR(gcn_opt,T_max=N_EPOCHS)
crit=nn.CrossEntropyLoss()
gcn_hist={k:[] for k in ['tl','vl','ta','va','vauc']}
best_gcn=0

print('Training GCN...')
for ep in tqdm(range(1,N_EPOCHS+1)):
    tl,ta=train_gcn(gcn_model,gcn_tr,gcn_opt,crit)
    vl,va,vauc,_,_=eval_gcn(gcn_model,gcn_vl,crit)
    gcn_sch.step()
    for k,v in zip(['tl','vl','ta','va','vauc'],[tl,vl,ta,va,vauc]): gcn_hist[k].append(v)
    if vauc>best_gcn:
        best_gcn=vauc; torch.save(gcn_model.state_dict(),f'{DRIVE_DIR}/best_gcn.pt')
    if ep%5==0: print(f'Ep{ep:3d} loss{tl:.4f}/{vl:.4f} acc{ta:.4f}/{va:.4f} AUC{vauc:.4f}')
print(f'Best GCN val AUC: {best_gcn:.4f}')

---
## 6. Architecture 2 — TreeNiN-style RvNN (Louppe et al. JHEP 2019)

### Graph construction
Pure numpy kt algorithm (p=−1, R=1.0) builds a binary tree per jet.
kt is IRC-safe; its merging history approximates the inverse QCD parton shower.
Trees are saved to Drive every 500 jets — resumable if session disconnects.

### Level-batching
Naive jet-by-jet recursion is O(N) sequential Python calls per jet — completely unusable on GPU.
Level-batching (Louppe et al. / diana-hep/TreeNiN): BFS collects nodes by depth, then all nodes
at the same depth across the entire batch are processed in ONE matrix operation.
This is what makes the TreeNiN paper's GPU training tractable.

### Model
- **Leaf NiN**: MLP(5 → h → h → h) — Network-in-Network, separate weights from inner nodes
- **Inner GRU**: GRUCell([h_left, h_right, split_feat] → h) — QCD-aware merge
- **Inner NiN**: MLP(h → h → h) — post-GRU refinement
- Root hidden state → classifier MLP(h → 2)

In [ ]:
def kt_cluster(feats):
    """Pure numpy kt clustering -> recursive tree dict."""
    N=len(feats)
    if N==1: return {'leaf':True,'feat':feats[0]}
    pt=np.exp(feats[:,0]).clip(1e-8)
    ea,pa,pta=feats[:,2].copy(),feats[:,3].copy(),pt.copy()
    active=list(range(N))
    nodes=[{'leaf':True,'feat':feats[i]} for i in range(N)]
    while len(active)>1:
        best,bi,bj=np.inf,None,None
        for ii in range(len(active)):
            for jj in range(ii+1,len(active)):
                i,j=active[ii],active[jj]
                de=ea[i]-ea[j]; dp=(pa[i]-pa[j]+np.pi)%(2*np.pi)-np.pi
                d=min(pta[i],pta[j])**2*(de**2+dp**2)
                if d<best: best,bi,bj=d,i,j
        if best<=min(pta[i]**2 for i in active):
            i,j=bi,bj; p1,p2=pta[i],pta[j]
            de=ea[i]-ea[j]; dp=(pa[i]-pa[j]+np.pi)%(2*np.pi)-np.pi
            sf=np.array([np.log(min(p1,p2)**2*(de**2+dp**2)+1e-8),
                         min(p1,p2)/(p1+p2+1e-8)],dtype=np.float32)
            pn=p1+p2
            ea[i]=(p1*ea[i]+p2*ea[j])/pn; pa[i]=(p1*pa[i]+p2*pa[j])/pn; pta[i]=pn
            nodes[i]={'leaf':False,'left':nodes[i],'right':nodes[j],'split_feat':sf}
            active.remove(j)
        else:
            active.remove(min(active,key=lambda x:pta[x]**2))
    return nodes[active[0]] if active else {'leaf':True,'feat':feats[0]}


def tree_to_arrays(node, content, tree_arr):
    """Convert recursive dict to index-based arrays for level-batching."""
    nid=len(content)
    if node['leaf']:
        content.append(('leaf', node['feat'])); tree_arr.append([-1,-1])
    else:
        content.append(('inner', node['split_feat'])); tree_arr.append([-1,-1])
        lid=tree_to_arrays(node['left'],  content, tree_arr)
        rid=tree_to_arrays(node['right'], content, tree_arr)
        tree_arr[nid]=[lid,rid]
    return nid


# ── Build trees (with Drive caching and resume support) ─────────────
TREE_CACHE=f'{DRIVE_DIR}/kt_trees_20k.pkl'
SAVE_EVERY=500

if os.path.exists(TREE_CACHE):
    print('Loading cached trees...')
    with open(TREE_CACHE,'rb') as f: raw_trees,labels_list=pickle.load(f)
    start=next((i for i,t in enumerate(raw_trees) if t is None),len(raw_trees))
    print(f'Loaded. Resume from {start}/{len(raw_trees)}')
else:
    raw_trees=[None]*len(valid); start=0
    print(f'Fresh start: {len(valid)} jets')

if start<len(valid):
    fail=0
    for i in tqdm(range(start,len(valid))):
        try: raw_trees[i]=kt_cluster(feats_list[i])
        except: raw_trees[i]=None; fail+=1
        if i%SAVE_EVERY==0 and i>0:
            with open(TREE_CACHE,'wb') as f: pickle.dump((raw_trees,labels_list),f)
    with open(TREE_CACHE,'wb') as f: pickle.dump((raw_trees,labels_list),f)
    print(f'Done. Failed:{fail}')
else:
    print('All trees ready.')

# Convert to index format
print('Converting to index-based format for level-batching...')
jet_dicts=[]; valid_labels=[]
for i,t in enumerate(tqdm(raw_trees)):
    if t is not None:
        c=[]; ta=[]; tree_to_arrays(t,c,ta)
        jet_dicts.append({'content':c,'tree':np.array(ta,dtype=np.int32),'root_id':0})
        valid_labels.append(labels_list[i])
print(f'Converted: {len(jet_dicts)} jets')

In [ ]:
def batch_by_levels(jet_batch, labels):
    """
    TreeNiN-style level batching (Louppe et al. 2017/2019, diana-hep/TreeNiN).
    BFS from root collects nodes by depth across the whole batch.
    Returns levels in reverse order (deepest first) for bottom-up processing.
    """
    all_levels=[]; pos_map={}

    for ji,jet in enumerate(jet_batch):
        tree=jet['tree']; content=jet['content']
        q=deque([(jet['root_id'],0)])
        while q:
            nid,depth=q.popleft()
            while len(all_levels)<=depth: all_levels.append([])
            pos=len(all_levels[depth])
            is_leaf=tree[nid][0]==-1
            all_levels[depth].append({
                'ji':ji,'nid':nid,'pos':pos,'is_leaf':is_leaf,
                'content':content[nid],
                'lid':int(tree[nid][0]),'rid':int(tree[nid][1])
            })
            pos_map[(ji,nid)]=(depth,pos)
            if not is_leaf:
                q.append((int(tree[nid][0]),depth+1))
                q.append((int(tree[nid][1]),depth+1))

    levels_rev=[]
    for depth in range(len(all_levels)-1,-1,-1):
        nodes=all_levels[depth]
        leaves=[n for n in nodes if n['is_leaf']]
        inners=[n for n in nodes if not n['is_leaf']]
        levels_rev.append({
            'depth':depth,'leaves':leaves,'inners':inners,
            'all_nodes':nodes,
            'left_refs': [(pos_map[(n['ji'],n['lid'])]) for n in inners],
            'right_refs':[(pos_map[(n['ji'],n['rid'])]) for n in inners],
        })
    return levels_rev, labels, pos_map, len(all_levels)

In [ ]:
class TreeNiN(nn.Module):
    """
    QCD-aware RvNN — adapted from Louppe, Cho, Cranmer et al. JHEP 2019
    and the diana-hep/TreeNiN implementation.

    Key insight: level-batching processes all nodes at the same tree depth
    across the entire batch simultaneously as one GPU matrix operation.

    Leaf NiN   : MLP(5->h->h->h)         — Network-in-Network, separate leaf weights
    Inner GRU  : GRUCell([h,h,sf]->h)     — QCD-aware merge with split features
    Inner NiN  : MLP(h->h->h)             — post-GRU refinement
    Classifier : MLP(h->64->2)
    """
    def __init__(self,h=64,drop=0.3):
        super().__init__()
        self.leaf_nin=nn.Sequential(
            nn.Linear(5,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU(),nn.Linear(h,h)
        )
        self.gru=nn.GRUCell(input_size=2*h+2,hidden_size=h)
        self.inner_nin=nn.Sequential(nn.Linear(h,h),nn.ReLU(),nn.Linear(h,h))
        self.drop=nn.Dropout(drop)
        self.clf=nn.Sequential(
            nn.Linear(h,64),nn.ReLU(),nn.Dropout(drop),nn.Linear(64,2)
        )

    def forward(self,levels_rev,labels,pos_map,n_levels,batch_size):
        node_out={}

        for ld in levels_rev:
            depth=ld['depth']; leaves=ld['leaves']; inners=ld['inners']

            # Batch all leaves at this depth in one MLP call
            if leaves:
                lf=torch.stack([
                    torch.tensor(n['content'][1],dtype=torch.float,device=DEVICE)
                    for n in leaves
                ])
                lh=self.leaf_nin(lf)
                for k,n in enumerate(leaves): node_out[(depth,n['pos'])]=lh[k]

            # Batch all inner nodes at this depth in one GRU call
            if inners:
                lh_c=torch.stack([node_out[r] for r in ld['left_refs']])
                rh_c=torch.stack([node_out[r] for r in ld['right_refs']])
                sf  =torch.stack([
                    torch.tensor(n['content'][1],dtype=torch.float,device=DEVICE)
                    for n in inners
                ])
                inp=torch.cat([lh_c,rh_c,sf],dim=1)
                h  =self.gru(inp,(lh_c+rh_c)/2)
                h  =self.inner_nin(h)
                h  =self.drop(h)
                for k,n in enumerate(inners): node_out[(depth,n['pos'])]=h[k]

        # Collect root states (depth=0, position = jet index)
        roots=torch.stack([node_out[(0,ji)] for ji in range(batch_size)])
        return self.clf(roots)


# Re-split on jet_dicts subset
n=len(jet_dicts)
perm=np.random.RandomState(SEED).permutation(n)
ntr=int(0.70*n); nvl=int(0.15*n)
jtr,jvl,jte=perm[:ntr].tolist(),perm[ntr:ntr+nvl].tolist(),perm[ntr+nvl:].tolist()

class TreeDataset(Dataset):
    def __init__(self,idx): self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=self.idx[i]; return jet_dicts[j],valid_labels[j]

def tree_col(batch):
    jets=[b[0] for b in batch]; labels=torch.tensor([b[1] for b in batch],dtype=torch.long)
    lv,lb,pm,nl=batch_by_levels(jets,labels)
    return lv,lb,pm,nl,len(jets)

TREE_BATCH=128
tree_tr=DataLoader(TreeDataset(jtr),TREE_BATCH,shuffle=True, collate_fn=tree_col,num_workers=0)
tree_vl=DataLoader(TreeDataset(jvl),TREE_BATCH,shuffle=False,collate_fn=tree_col,num_workers=0)
tree_te=DataLoader(TreeDataset(jte),TREE_BATCH,shuffle=False,collate_fn=tree_col,num_workers=0)

rvnn_model=TreeNiN(h=64,drop=0.3).to(DEVICE)
print(f'TreeNiN params: {sum(p.numel() for p in rvnn_model.parameters()):,}')
print(f'Train batches: {len(tree_tr)}')

In [ ]:
def train_tree(model,loader,opt,crit):
    model.train(); tl=tc=tt=0
    for lv,lb,pm,nl,bs in loader:
        lb=lb.to(DEVICE); opt.zero_grad()
        lg=model(lv,lb,pm,nl,bs); loss=crit(lg,lb); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        tl+=loss.item()*bs; tc+=(lg.argmax(1)==lb).sum().item(); tt+=bs
    return tl/tt,tc/tt

@torch.no_grad()
def eval_tree(model,loader,crit):
    model.eval(); tl=tc=tt=0; P,L=[],[]
    for lv,lb,pm,nl,bs in loader:
        lb=lb.to(DEVICE)
        lg=model(lv,lb,pm,nl,bs); loss=crit(lg,lb)
        tl+=loss.item()*bs; tc+=(lg.argmax(1)==lb).sum().item(); tt+=bs
        P.append(F.softmax(lg,1)[:,1].cpu()); L.append(lb.cpu())
    P=torch.cat(P).numpy(); L=torch.cat(L).numpy()
    return tl/tt,tc/tt,roc_auc_score(L,P),P,L

rv_opt=torch.optim.AdamW(rvnn_model.parameters(),lr=3e-4,weight_decay=1e-4)
rv_sch=torch.optim.lr_scheduler.CosineAnnealingLR(rv_opt,T_max=N_EPOCHS)
rv_crit=nn.CrossEntropyLoss()
rv_hist={k:[] for k in ['tl','vl','ta','va','vauc']}
best_rv=0

print('Training TreeNiN RvNN (level-batched)...')
for ep in tqdm(range(1,N_EPOCHS+1)):
    tl,ta=train_tree(rvnn_model,tree_tr,rv_opt,rv_crit)
    vl,va,vauc,_,_=eval_tree(rvnn_model,tree_vl,rv_crit)
    rv_sch.step()
    for k,v in zip(['tl','vl','ta','va','vauc'],[tl,vl,ta,va,vauc]): rv_hist[k].append(v)
    if vauc>best_rv:
        best_rv=vauc; torch.save(rvnn_model.state_dict(),f'{DRIVE_DIR}/best_rvnn.pt')
    if ep%5==0: print(f'Ep{ep:3d} loss{tl:.4f}/{vl:.4f} acc{ta:.4f}/{va:.4f} AUC{vauc:.4f}')
print(f'Best RvNN val AUC: {best_rv:.4f}')

---
## 7. Evaluation & Plots

In [ ]:
gcn_model.load_state_dict(torch.load(f'{DRIVE_DIR}/best_gcn.pt',  map_location=DEVICE))
rvnn_model.load_state_dict(torch.load(f'{DRIVE_DIR}/best_rvnn.pt', map_location=DEVICE))

_,ga,gau,gp,gl=eval_gcn(gcn_model,  gcn_te,  crit)
_,ra,rau,rp,rl=eval_tree(rvnn_model, tree_te, rv_crit)

print('='*52)
print(f'  GCN     — Acc: {ga:.4f}  AUC: {gau:.4f}')
print(f'  TreeNiN — Acc: {ra:.4f}  AUC: {rau:.4f}')
print('='*52)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))

gf,gt,_=roc_curve(gl,gp); rf,rt,_=roc_curve(rl,rp)
axes[0].plot(gt,np.where(gf>0,1/gf,0),color='steelblue', lw=2,label=f'GCN AUC={gau:.4f}')
axes[0].plot(rt,np.where(rf>0,1/rf,0),color='darkorange',lw=2,label=f'TreeNiN AUC={rau:.4f}')
axes[0].set_yscale('log'); axes[0].set_xlim([0,1]); axes[0].grid(True,alpha=0.3)
axes[0].set_xlabel('Quark efficiency'); axes[0].set_ylabel('Gluon rejection (1/FPR)')
axes[0].set_title('ROC — background rejection'); axes[0].legend()

ep=range(1,N_EPOCHS+1)
axes[1].plot(ep,gcn_hist['tl'], color='steelblue', ls='-',  label='GCN train')
axes[1].plot(ep,gcn_hist['vl'], color='steelblue', ls='--', label='GCN val')
axes[1].plot(ep,rv_hist['tl'],  color='darkorange',ls='-',  label='RvNN train')
axes[1].plot(ep,rv_hist['vl'],  color='darkorange',ls='--', label='RvNN val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Learning curves'); axes[1].legend(); axes[1].grid(True,alpha=0.3)

axes[2].plot(ep,gcn_hist['vauc'], color='steelblue', lw=2,label='GCN')
axes[2].plot(ep,rv_hist['vauc'],  color='darkorange',lw=2,label='TreeNiN RvNN')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val AUC')
axes[2].set_title('AUC progression'); axes[2].legend(); axes[2].grid(True,alpha=0.3)

plt.suptitle('GCN vs TreeNiN RvNN — Quark/Gluon Classification',fontsize=14,y=1.01)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/performance.png',dpi=150,bbox_inches='tight')
plt.show()

## 8. Discussion

### GCN
The GCN builds a **data-driven geometric graph**: nodes connected by spatial proximity in
(Δη, Δφ). kNN connectivity captures local clustering and subjet structure without assuming
a shower history. Dual global pooling retains both average and extremal particle features.
Fully GPU-parallelisable and permutation-invariant.

**Limitation:** kNN topology is not IRC safe — soft/collinear splittings change the graph
structure, raising reliability concerns for real detector data (Dreyer & Granik 2021).

### TreeNiN RvNN (Louppe et al. JHEP 2019)
The RvNN imposes a **physics-motivated hierarchical structure** via kt jet clustering.
kt is IRC-safe; the merging history approximates the inverse QCD parton shower.
The GRU cell at internal nodes uses splitting features (log kt distance, momentum fraction z)
that are themselves IRC-safe QCD observables — making the model explicitly QCD-aware.
Gluon jets (Ca=3 > Cf=4/3) produce more symmetric, lower-z splittings; differences the
RvNN exploits at every level of the tree.

**Level-batching** is the key engineering contribution that makes this practical: rather than
processing one tree at a time, all nodes at the same depth across the batch are processed
as one GPU matrix operation — the approach used in the original diana-hep/TreeNiN codebase.

### Comparison

| | GCN | TreeNiN RvNN |
|---|---|---|
| Graph construction | kNN in (Δη, Δφ) | kt clustering tree |
| IRC safety | Partial | Yes |
| Physics bias | Weak (geometric proximity) | Strong (QCD shower hierarchy) |
| Permutation invariant | Yes | Yes |
| GPU acceleration | Full batching | Level-batching |
| Interpretability | Moderate | High (split feats ↔ QCD obs.) |